In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from groq import Groq

In [2]:
load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_yp') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'openai/gpt-oss-120b'
groq = Groq()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [7]:
def select_relevant_links(url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 9 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'jobs application',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'social media', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 6 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
5 days ago
•
51.9k
•
2.29k
black-forest-labs/flux-3-image-klein-comfy
Updated
5 days ago
•
513k
•
1.25k
meta-models/Muse-Spark-Open-816B-A42B
Updated
7 days ago
•
3.16k
•
651
black-forest-labs/flux-3-image-klein
Updated
6 days ago
•
103k
•
623
OpenMOSS-Team/MOSS-Image-SFT
Updated
1 day ago
•
6.89k
•
552
Browse 2M+ models
Spaces
Running


In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 7 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated\n5 days ago\n•\n51.9k\n•\n2.29k\nblack-forest-labs/flux-3-image-klein-

In [17]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 6 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


**Hugging Face – The AI Community Building the Future**  

---

## 1️⃣ Who We Are  
Hugging Face is the world’s largest collaborative hub for machine‑learning (ML) research and production.  Our platform lets developers, data scientists, and researchers **create, discover, and share** AI models, datasets, and applications at massive scale—today more than **2 M+ models**, **500 k+ datasets**, and **1 M+ ready‑to‑run applications**.  

We power the open‑source AI movement while delivering **enterprise‑grade** services that let businesses bring cutting‑edge models to production securely and at speed.

---

## 2️⃣ What We Offer  

| Offering | Key Benefits | Ideal Users |
|----------|--------------|-------------|
| **Models** | Browse, fine‑tune, and host 2 M+ community models (LLMs, vision, audio, RL, diffusion, etc.) | Researchers, product teams |
| **Datasets** | 500 k+ curated, versioned datasets with built‑in licensing and metadata | Data engineers, scientists |
| **Spaces** | Deploy interactive ML apps (chatbots, image generators, video tools) with one‑click hosting | Developers, demo creators |
| **Buckets & Storage** | Scalable, secure object storage for large model weights & datasets | Enterprises, MLOps teams |
| **Inference Endpoints** | Fully managed, low‑latency API for any model, with auto‑scaling & SLA options | SaaS platforms, fintech, health, media |
| **Enterprise Solutions** | PRO plans, dedicated support, private repositories, compliance & audit trails | Fortune 500, regulated industries |
| **Community & Learning** | Discord, Forum, Blog, Daily Papers, tutorials, and a vibrant open‑source contributor base | Everyone who wants to stay on the AI frontier |

---

## 3️⃣ Why Customers Choose Hugging Face  

* **Speed to market** – Deploy a model in minutes via Inference Endpoints or Spaces.  
* **Reliability** – Enterprise‑grade SLAs, multi‑region redundancy, and 24/7 support.  
* **Security & Compliance** – Private repos, role‑based access, GDPR/CCPA‑ready data handling.  
* **Ecosystem Integration** – Native connectors for AWS, Azure, GCP, Kubernetes, and popular ML frameworks (PyTorch, TensorFlow, JAX).  
* **Cost Transparency** – Pay‑as‑you‑go pricing, with volume discounts for large workloads.  

> *“Hugging Face turned our research prototype into a production API overnight. The community models gave us a 30 % accuracy boost without extra data collection.”* – Senior AI Engineer, Global FinTech Firm  

---

## 4️⃣ Culture & Community  

* **Open‑source first** – All core libraries (Transformers, Diffusers, Datasets, Tokenizers) live on GitHub under permissive licenses.  
* **Collaboration at heart** – Thousands of contributors worldwide co‑author models, papers, and tutorials.  
* **Diversity & Inclusion** – Remote‑first workplace, employee‑resource groups, and a strong commitment to ethical AI.  
* **Learning mindset** – Daily Papers, weekly webinars, and a thriving Discord/Forum where members mentor each other.  

Our mantra: **“Build together, ship together.”**  

---

## 5️⃣ Careers – Join the Mission  

Hugging Face is expanding rapidly. Current openings span:

| Role | Location / Remote | What You’ll Do |
|------|-------------------|----------------|
| Machine Learning Engineer | Remote / Global | Build and optimize large‑scale inference pipelines |
| Research Scientist (NLP / CV / Audio) | Remote | Advance state‑of‑the‑art models and publish open‑source work |
| Solutions Engineer – Enterprise | Remote (US/EU) | Partner with customers to design custom AI deployments |
| Community Manager | Remote | Grow and support the global contributor ecosystem |
| Product Manager – Platform | Remote | Define roadmap for Models, Spaces, and Buckets |

*Why work with us?*  
- **Impact:** Your code reaches millions of developers instantly.  
- **Growth:** Direct mentorship from world‑leading AI researchers.  
- **Benefits:** Competitive salaries, equity, flexible hours, and a generous learning budget.  

Apply now at **huggingface.co/careers**.

---

## 6️⃣ Investment Highlights  

* **Market Position:** The de‑facto hub for over 2 M ML models – a network effect that fuels continuous growth.  
* **Revenue Streams:** Subscription (PRO), Enterprise contracts, Inference usage fees, and premium support.  
* **Scalable Architecture:** Cloud‑agnostic infrastructure that can serve billions of requests per year.  
* **Strong Brand Equity:** Recognized by top tech media and trusted by leading enterprises (e.g., Fortune 500, major research labs).  
* **Future Roadmap:** Expansion of multimodal models, AI‑generated content tools, and tighter integration with edge devices.

---

## 7️⃣ Get in Touch  

**Website:** https://huggingface.co  
**Docs & API:** https://huggingface.co/docs  
**Community:** Discord, Forum, GitHub, Twitter  
**Enterprise Sales:** sales@huggingface.co  
**Press & Partnerships:** press@huggingface.co  

*Join the AI revolution. Build, share, and monetize together with Hugging Face.*


With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 5 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your gateway to open‑source machine learning, collaborative innovation, and enterprise‑grade AI solutions.*

---  

## 📌 Company Overview  
- **Founded:** 2016, New York & Paris  
- **Mission:** Democratize AI by providing a unified platform where researchers, developers, and enterprises can **create, share, and deploy** models, datasets, and applications.  
- **Core Values:** Openness, collaboration, inclusivity, and responsible AI.  

---

## 🧩 What We Offer  

| Product | What It Does | Typical Users |
|--------|--------------|----------------|
| **Models** | >2 M open‑source models for vision, language, speech, and multimodal tasks. | Researchers, startups, product teams |
| **Datasets** | 500 k+ curated datasets covering text, images, video, finance, cybersecurity, etc. | Data scientists, ML engineers |
| **Spaces** | Interactive demo‑hosted apps that run in the browser with zero‑setup. | Developers, educators, community builders |
| **Buckets** | Scalable storage for large model artefacts and training data. | Enterprises, research labs |
| **Inference Endpoints & Providers** | Managed, low‑latency serving of models with enterprise SLAs. | SaaS products, B2B services |
| **Enterprise Solutions** | Private clouds, compliance tooling, dedicated support, and custom integrations. | Fortune‑500, regulated industries |
| **Hugging Face PRO** | Premium features – priority support, advanced analytics, and early‑access tools. | Growing AI teams, consultancies |

---

## 🌍 Community & Culture  

- **Open‑source first:** Every model, dataset, and Space is public unless explicitly private.  
- **Collaboration hub:** Over **108 k AI & ML followers** and a vibrant Discord, Forum, and GitHub community.  
- **Inclusive environment:** Remote‑friendly, diverse hiring practices, and regular community events (hackathons, webinars, “Daily Papers” newsletter).  
- **Transparency:** Activity feeds showcase real‑time contributions from users worldwide.  
- **Responsible AI:** Dedicated teams work on bias mitigation, data privacy, and ethical guidelines.

---

## 👥 Who Uses Hugging Face?  

- **Academic & research labs** – publish and benchmark state‑of‑the‑art models.  
- **Start‑ups** – accelerate product development with ready‑to‑use APIs and Spaces.  
- **Enterprises** – finance, healthcare, media, and tech giants run secure inference workloads on Hugging Face Enterprise.  
- **Developers & hobbyists** – explore 2 M+ models, build demos, and share knowledge.  

*Examples (publicly mentioned):*  
- **Meta**, **Microsoft**, **NVIDIA**, **Amazon**, **Google** (model hosting & research collaborations).  
- **FinTech & trading firms** using the *venvoo/china-a-share* dataset for market analysis.  
- **Content platforms** leveraging *datasocial/tiktok-5.6B-videos* for video understanding.  

---

## 🚀 Careers & Growth  

| Area | What You’ll Do | Why It Matters |
|------|----------------|----------------|
| **Research Engineering** | Build next‑gen transformer architectures; publish papers; open‑source breakthroughs. | Push the frontier of AI. |
| **Product & Design** | Shape the user experience of Models, Spaces, and Enterprise dashboards. | Make AI accessible to all skill levels. |
| **Community & Partnerships** | Grow the global HF ecosystem; run events; support open‑source contributors. | Foster a thriving, collaborative culture. |
| **Infrastructure & Security** | Scale storage buckets, inference endpoints, and ensure compliance. | Deliver reliable, trustworthy AI services. |
| **Responsible AI** | Develop bias detection tools, data governance policies, and ethical frameworks. | Keep AI safe and fair. |

- **Work model:** Mostly remote, with hubs in New York, Paris, and London.  
- **Benefits:** Competitive salary, equity, learning stipend, health & wellness programs, and generous open‑source contribution time.  
- **Hiring portal:** Visit the **Team & Enterprise** section on huggingface.co for current openings.

---

## 📞 Get in Touch  

- **Website:** https://huggingface.co  
- **Community:** Discord, Forum, GitHub, and the **HuggingChat** AI assistant.  
- **Enterprise Sales:** Contact via the **Enterprise** page for demos, pricing, and custom solutions.  

---

*Join a community where **collaboration fuels innovation**, and where every line of code can shape the future of artificial intelligence.*

In [ ]:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 6 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your partner for open, collaborative, and enterprise‑ready machine learning.*

---

## 🌟 Who We Are  

Hugging Face is the world’s leading collaboration platform for the machine‑learning (ML) ecosystem. Our mission is to make AI development **open, accessible, and trustworthy** by giving researchers, developers, and businesses a single hub to share models, datasets, and applications.  

- **2 M+** pre‑trained models spanning vision, language, audio, and multimodal tasks.  
- **500 k+** high‑quality datasets covering everything from social media to finance.  
- **1 M+** ready‑to‑run AI applications (Spaces) that turn research into products in minutes.  

We power the next generation of AI products, from startups to Fortune 500 enterprises, and nurture a vibrant community of more than **1 M** active contributors.

---

## 🛠️ Core Offerings  

| Offering | What It Does | Why It Matters |
|----------|--------------|----------------|
| **Models Hub** | Browse, host, and version‑control 2 M+ models; instant download via `transformers` and `diffusers`. | Accelerates R&D, reduces duplication, and ensures reproducibility. |
| **Datasets Hub** | Search, filter, and stream 500 k+ datasets with built‑in licensing metadata. | Enables ethical data sourcing and rapid prototyping. |
| **Spaces** | Deploy interactive demos (Gradio or Streamlit) with a single click; supports GPU/CPU inference. | Turns research into user‑facing products without DevOps overhead. |
| **Inference Endpoints & Providers** | Managed, scalable serving of models with pay‑as‑you‑go pricing; private endpoints for enterprise security. | Guarantees low‑latency, production‑grade AI at scale. |
| **Storage Buckets** | Secure object storage for large model weights, datasets, and logs. | Keeps your assets organized and compliant. |
| **Enterprise Solutions** | Dedicated support, SLAs, on‑prem deployment, and custom integration services. | Aligns AI initiatives with corporate governance and ROI goals. |
| **Hugging Face PRO** | Premium tools, priority support, and early‑access features for power users. | Boosts productivity for teams that need the edge. |

---

## 🤝 Community & Culture  

- **Open Collaboration:** Every model, dataset, and Space is versioned and openly licensed (when permissible), encouraging reuse and collective improvement.  
- **Transparency & Ethics:** We surface licensing, provenance, and data‑card information to help users make responsible AI choices.  
- **Inclusive Environment:** A multilingual, globally distributed community interacts via Discord, Forum, GitHub, and regular virtual meet‑ups.  
- **Learning First:** The **Learn** hub, daily papers digest, and extensive documentation empower newcomers and seasoned researchers alike.  

Our brand colors—#FFD21E, #FF9D00, #6B7280—reflect a vibrant yet professional identity, and the iconic “🤗” logo symbolizes our welcoming, collaborative spirit.

---

## 📊 Who Uses Hugging Face  

| Segment | Typical Use Cases |
|---------|-------------------|
| **Startups & SaaS** | Fast‑track AI product launches with Spaces and hosted inference. |
| **Enterprise AI Teams** | Secure model hosting, compliance‑ready datasets, and dedicated support for mission‑critical workloads. |
| **Academic & Research Labs** | Share reproducible research artifacts; benchmark against 2 M+ community models. |
| **Government & NGOs** | Leverage open‑source models for public‑good projects while respecting data‑privacy regulations. |

*Examples include large language model (LLM) deployment for customer support, computer‑vision models for medical imaging, and financial‑time‑series datasets for algorithmic trading.*

---

## 🚀 Careers & Growth  

Hugging Face is constantly expanding its talent pool. We look for people who:

- **Love Open Source** – contribute to libraries like `transformers`, `datasets`, and `accelerate`.  
- **Thrive on Collaboration** – enjoy cross‑functional teamwork with researchers, product managers, and engineers worldwide.  
- **Champion Responsible AI** – advocate for fairness, transparency, and sustainability in ML.  

Open positions span **Engineering, Product, Research, Community Management, and Sales** (Enterprise & Partnerships). Join us to shape the future of AI while working in a flexible, remote‑first environment that values learning and impact.

---

## 📞 Get in Touch  

- **Website:** https://huggingface.co  
- **Community:** Discord, Forum, and GitHub (open‑source contributions welcome)  
- **Enterprise Inquiries:** [Enterprise Support] → dedicated account managers and SLA options.  
- **Press & Brand Assets:** Download official logos, color palettes, and guidelines from the **Brand Assets** page.  

---

**Hugging Face** – where the world’s AI talent collaborates, innovates, and builds the future together.  

*Explore AI Apps, browse models, or start a Space today – the hub for every machine‑learning journey.*